# Evaluate trained PCT vs heuristics

Comprehensive benchmarking suite for the PCT model trained in `01_train_pct.ipynb`.
Produces all the tables / plots needed for the thesis chapter:
1. Macro benchmark (Alexandria-realistic + BR mix)
2. Voyage-size sweep (50 / 100 / 200 items)
3. Container-type sweep (20GP / 40GP / 40HC / 45HC)
4. BR1 academic benchmark (compares directly to GOPT 76% reference)
5. Edge cases (hazmat, all-reefer, all-machinery, single-giant)
6. CoG analysis
7. Inference timing
8. 3D visual gallery (best / median / worst voyage)
9. Final thesis summary CSV + one-page figure

## 1. Setup + load model

In [ ]:
import os, sys, subprocess
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
REPO_URL = 'https://github.com/Seif-Sameh/loading-service-2.git'
BRANCH = 'main'
if os.path.isdir('loading-service-2'):
    subprocess.run(['rm', '-rf', 'loading-service-2'], check=True)
subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, 'loading-service-2'], check=True)
os.chdir('loading-service-2')
if os.getcwd() not in sys.path: sys.path.insert(0, os.getcwd())
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--upgrade-strategy', 'only-if-needed',
    '-r', 'requirements.txt', '-r', 'requirements-rl.txt',
], check=True)
if not os.path.isdir('/tmp/wadaboa-bpp'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Wadaboa/3d-bpp.git', '/tmp/wadaboa-bpp'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.prepare_datasets',
                '--wadaboa-pkl', '/tmp/wadaboa-bpp/data/products.pkl'], check=True)
print('environment ready')

In [ ]:
import shutil, glob
import torch
MODEL_PATH_OVERRIDE = None  # set explicitly if you prefer

def _find_model():
    if MODEL_PATH_OVERRIDE and os.path.isfile(MODEL_PATH_OVERRIDE): return MODEL_PATH_OVERRIDE
    cands = sorted(glob.glob('/kaggle/input/**/*.pt', recursive=True))
    if cands:
        os.makedirs('models', exist_ok=True)
        target = f'models/{os.path.basename(cands[0])}'
        if not os.path.isfile(target): shutil.copy(cands[0], target)
        return target
    local = sorted(glob.glob('models/*.pt'))
    if local: return local[0]
    raise FileNotFoundError('No .pt found. Upload via Kaggle Add Data -> Upload, or set MODEL_PATH_OVERRIDE.')

MODEL_PATH = _find_model()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using {MODEL_PATH}  ({os.path.getsize(MODEL_PATH)/1024:.1f} KB) on {device}')

In [ ]:
import time, statistics, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from app.catalog.loader import get_container, get_cargo_preset
from app.data.alexandria_sampler import AlexandriaSampler, SamplerConfig
from app.data.br_loader import (
    br_container_to_isolike, br_problem_to_items, list_br_problems,
)
from app.algorithms import get_algorithm
from app.algorithms.base import solve
from app.algorithms.pct.pct_agent import PCTPackingAgent
from app.schemas import CargoItem, Dimensions, FragilityClass, HazmatClass

PALETTE = {
    'bl':              '#1a4d7a', 'extreme_points':  '#2a6aa6', 'baf':  '#5a8bc9',
    'bssf':            '#94b8d9', 'blsf':             '#c5d8e6', 'pct':  '#2e7d32',
}
ALG_LABELS = {
    'bl': 'Bottom-Left', 'extreme_points': 'Extreme Points', 'baf': 'Best Area Fit',
    'bssf': 'Best Shortest Side', 'blsf': 'Best Longest Side',
    'pct': 'PCT (ours)',
}
ALL_ALGS = ['bl', 'extreme_points', 'baf', 'bssf', 'blsf', 'pct']
OUT = Path('benchmarks/out'); OUT.mkdir(parents=True, exist_ok=True)

PCT = PCTPackingAgent(weights_path=MODEL_PATH, device=device)
def get_alg(code): return PCT if code == 'pct' else get_algorithm(code)
print('ready -- algorithms:', list(ALG_LABELS.keys()))

## 2. Macro benchmark (Alexandria mixed, 100 items, 40HC)

In [ ]:
N_VOYAGES = 20
ITEMS = 100
alex = AlexandriaSampler(SamplerConfig(n_items=ITEMS, strategy='mixed', seed=None))
_macro = [(get_container('40HC'), alex.sample()) for _ in range(N_VOYAGES)]

def run_suite(voyages, algs):
    rows = []
    for vi, (c, its) in enumerate(voyages):
        for code in algs:
            res, _ = solve(algorithm=get_alg(code), container=c, items=its)
            k = res.kpis
            rows.append({
                'voyage': vi, 'algorithm': code, 'container': c.code.value,
                'n_items': len(its), 'placed': len(res.placements),
                'placed_pct': 100*len(res.placements)/max(len(its),1),
                'util_pct': 100*k.utilization, 'weight_pct': 100*k.weight_used,
                'cog_long_dev': k.cog_long_dev, 'cog_lat_dev': k.cog_lat_dev,
                'cog_vert_frac': k.cog_vert_frac,
                'unstable': k.unstable_count, 'imdg': k.imdg_violation_count,
                'lifo': k.lifo_violation_count, 'stack': k.stack_violation_count,
                'overloaded': k.overloaded_count, 'elapsed_ms': res.elapsed_ms,
            })
    return pd.DataFrame(rows)

macro_df = run_suite(_macro, ALL_ALGS)
macro_df.to_csv(OUT/'macro.csv', index=False)
agg = macro_df.groupby('algorithm').agg(
    util_mean=('util_pct','mean'), util_std=('util_pct','std'),
    placed_mean=('placed_pct','mean'), ms=('elapsed_ms','mean')
).reindex(ALL_ALGS)
agg.index = [ALG_LABELS[c] for c in agg.index]
print(agg.round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(len(ALL_ALGS))
means = [macro_df[macro_df.algorithm==a]['util_pct'].mean() for a in ALL_ALGS]
stds  = [macro_df[macro_df.algorithm==a]['util_pct'].std()  for a in ALL_ALGS]
axes[0].bar(x, means, yerr=stds, color=[PALETTE[a] for a in ALL_ALGS], alpha=0.85, capsize=5)
axes[0].set_xticks(x); axes[0].set_xticklabels([ALG_LABELS[a] for a in ALL_ALGS], rotation=15, fontsize=8)
axes[0].set_ylabel('util %'); axes[0].set_title(f'Macro: {N_VOYAGES} voyages x {ITEMS} items')
for xi, m in zip(x, means): axes[0].text(xi, m+0.5, f'{m:.1f}', ha='center', fontsize=8, fontweight='bold')
data = [macro_df[macro_df.algorithm==a]['util_pct'].values for a in ALL_ALGS]
bp = axes[1].boxplot(data, patch_artist=True, labels=[ALG_LABELS[a].replace(' ','\n') for a in ALL_ALGS])
for patch, code in zip(bp['boxes'], ALL_ALGS):
    patch.set_facecolor(PALETTE[code]); patch.set_alpha(0.85)
axes[1].set_ylabel('util %'); axes[1].set_title('Distribution per algorithm')
axes[1].tick_params(axis='x', labelsize=8); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(OUT/'macro.png', dpi=130, bbox_inches='tight'); plt.show()

## 3. BR academic benchmark (direct comparison to GOPT 76% reference)

In [ ]:
BR = list_br_problems()
N_BR = 20
br_voyages = []
for p in BR[:N_BR]:
    br_voyages.append((br_container_to_isolike(p), br_problem_to_items(p)))
br_df = run_suite(br_voyages, ALL_ALGS)
br_df.to_csv(OUT/'br.csv', index=False)
br_agg = br_df.groupby('algorithm').agg(util=('util_pct','mean'), placed=('placed_pct','mean')).reindex(ALL_ALGS)
br_agg.index = [ALG_LABELS[c] for c in br_agg.index]
print(br_agg.round(2).to_string())

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(ALL_ALGS))
ax.bar(x, br_agg['util'].values, color=[PALETTE[a] for a in ALL_ALGS], alpha=0.85)
ax.axhline(76, color='red', linestyle='--', alpha=0.5, label='GOPT (paper) ~76%')
ax.axhline(60.5, color='gray', linestyle=':', alpha=0.5, label='Best heuristic (DBL paper) ~60.5%')
ax.set_xticks(x); ax.set_xticklabels([ALG_LABELS[a] for a in ALL_ALGS], rotation=15, fontsize=8)
ax.set_ylabel('util %'); ax.set_title(f'BR1 academic benchmark - {N_BR} problems')
for xi, m in zip(x, br_agg['util'].values): ax.text(xi, m+1, f'{m:.1f}', ha='center', fontsize=8, fontweight='bold')
ax.legend()
plt.tight_layout(); plt.savefig(OUT/'br.png', dpi=130, bbox_inches='tight'); plt.show()

## 4. Voyage-size sweep + container-type sweep + edge cases

All in one cell to keep timing budgets in check.

In [ ]:
# size sweep
size_dfs = []
for n in [50, 100, 200]:
    s = AlexandriaSampler(SamplerConfig(n_items=n, strategy='mixed', seed=None))
    voys = [(get_container('40HC'), s.sample()) for _ in range(6)]
    df = run_suite(voys, ALL_ALGS); df['size'] = n; size_dfs.append(df)
size_df = pd.concat(size_dfs, ignore_index=True)
size_df.to_csv(OUT/'size_sweep.csv', index=False)

# container sweep
cont_dfs = []
for code in ['20GP', '40GP', '40HC', '45HC']:
    s = AlexandriaSampler(SamplerConfig(n_items=80, strategy='mixed', seed=None))
    voys = [(get_container(code), s.sample()) for _ in range(5)]
    df = run_suite(voys, ALL_ALGS); cont_dfs.append(df)
container_df = pd.concat(cont_dfs, ignore_index=True)
container_df.to_csv(OUT/'container_sweep.csv', index=False)

# edge cases
def edge_pure_real(seed):
    s = AlexandriaSampler(SamplerConfig(n_items=120, strategy='real', seed=seed))
    return get_container('40HC'), s.sample()
def edge_hazmat(seed):
    rng = random.Random(seed); items = []
    for i in range(80):
        if rng.random() < 0.5: items.append(get_cargo_preset('hazmat_corrosive_drum', f'haz-{i}'))
        else: items.append(get_cargo_preset(rng.choice(['eur_pallet_light', 'carton_large']), f'r-{i}'))
    return get_container('40HC'), items
def edge_machinery(seed):
    return get_container('40HC'), [get_cargo_preset('machinery_small', f'm-{i}') for i in range(20)]
def edge_reefer(seed):
    return get_container('40RF'), [get_cargo_preset('reefer_fruit_pallet', f'r-{i}') for i in range(40)]
EDGE = [('pure_real', edge_pure_real), ('hazmat', edge_hazmat), ('machinery', edge_machinery), ('reefer', edge_reefer)]
edge_dfs = []
for name, fn in EDGE:
    voys = [fn(s) for s in range(3)]
    df = run_suite(voys, ALL_ALGS); df['edge_case'] = name; edge_dfs.append(df)
edge_df = pd.concat(edge_dfs, ignore_index=True)
edge_df.to_csv(OUT/'edge.csv', index=False)
print('done all sweeps')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for code in ALL_ALGS:
    sub = size_df[size_df.algorithm==code].groupby('size')['util_pct'].mean()
    axes[0].plot(sub.index, sub.values, 'o-', label=ALG_LABELS[code], color=PALETTE[code])
axes[0].set_xlabel('items per voyage'); axes[0].set_ylabel('util %'); axes[0].set_title('Voyage-size sweep'); axes[0].legend(fontsize=7)

ind = np.arange(4); width = 1.0/(len(ALL_ALGS)+1)
for i, code in enumerate(ALL_ALGS):
    means = [container_df[(container_df.algorithm==code) & (container_df.container==c)]['util_pct'].mean()
             for c in ['20GP','40GP','40HC','45HC']]
    axes[1].bar(ind + i*width, means, width, label=ALG_LABELS[code], color=PALETTE[code])
axes[1].set_xticks(ind + width*(len(ALL_ALGS)-1)/2)
axes[1].set_xticklabels(['20GP','40GP','40HC','45HC'])
axes[1].set_ylabel('util %'); axes[1].set_title('Container sweep'); axes[1].legend(fontsize=7)

pivot = edge_df.groupby(['edge_case','algorithm'])['util_pct'].mean().unstack().reindex(columns=ALL_ALGS)
pivot.columns = [ALG_LABELS[c].split()[0] for c in pivot.columns]
im = axes[2].imshow(pivot.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=70)
axes[2].set_xticks(range(len(pivot.columns))); axes[2].set_xticklabels(pivot.columns, rotation=15, fontsize=8)
axes[2].set_yticks(range(len(pivot.index))); axes[2].set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        axes[2].text(j, i, f'{pivot.values[i,j]:.0f}', ha='center', va='center', fontsize=8)
axes[2].set_title('Edge cases (util %)')
plt.tight_layout(); plt.savefig(OUT/'sweeps_and_edges.png', dpi=130, bbox_inches='tight'); plt.show()

## 5. 3D gallery (best / median / worst voyage)

In [ ]:
pct_macro = macro_df[macro_df.algorithm=='pct'].sort_values('util_pct').reset_index()
best_idx = int(pct_macro.iloc[-1]['voyage'])
med_idx  = int(pct_macro.iloc[len(pct_macro)//2]['voyage'])
worst_idx = int(pct_macro.iloc[0]['voyage'])

def draw_box(ax, x, y, z, dx, dy, dz, color):
    verts = [
        [(x,y,z),(x+dx,y,z),(x+dx,y+dy,z),(x,y+dy,z)],
        [(x,y,z+dz),(x+dx,y,z+dz),(x+dx,y+dy,z+dz),(x,y+dy,z+dz)],
        [(x,y,z),(x+dx,y,z),(x+dx,y,z+dz),(x,y,z+dz)],
        [(x,y+dy,z),(x+dx,y+dy,z),(x+dx,y+dy,z+dz),(x,y+dy,z+dz)],
        [(x,y,z),(x,y+dy,z),(x,y+dy,z+dz),(x,y,z+dz)],
        [(x+dx,y,z),(x+dx,y+dy,z),(x+dx,y+dy,z+dz),(x+dx,y,z+dz)],
    ]
    ax.add_collection3d(Poly3DCollection(verts, alpha=0.55, facecolor=color, edgecolor='black', linewidth=0.2))

def render(ax, container, items, code, label):
    res, _ = solve(algorithm=get_alg(code), container=container, items=items)
    cs = plt.cm.viridis(np.linspace(0, 1, max(len(res.placements), 1)))
    for i, p in enumerate(res.placements):
        draw_box(ax, p.position.x_mm, p.position.z_mm, p.position.y_mm,
                 p.rotated_dimensions.length_mm, p.rotated_dimensions.width_mm, p.rotated_dimensions.height_mm, cs[i])
    ax.set_xlim(0, container.internal.length_mm); ax.set_ylim(0, container.internal.width_mm); ax.set_zlim(0, container.internal.height_mm)
    ax.set_box_aspect([container.internal.length_mm, container.internal.width_mm, container.internal.height_mm])
    ax.set_title(f'{label}\n{ALG_LABELS[code]}: util {res.kpis.utilization*100:.1f}%, placed {len(res.placements)}/{len(items)}', fontsize=9)

fig = plt.figure(figsize=(15, 10))
for row, (vidx, label) in enumerate([(best_idx,'BEST'), (med_idx,'MEDIAN'), (worst_idx,'WORST')]):
    cont, items = _macro[vidx]
    for col, code in enumerate(['bl', 'extreme_points', 'pct']):
        ax = fig.add_subplot(3, 3, row*3 + col + 1, projection='3d')
        render(ax, cont, items, code, label if col == 0 else '')
fig.suptitle('Best / Median / Worst (by PCT util)  -  3 algorithms each', fontweight='bold', y=1.0)
plt.tight_layout(); plt.savefig(OUT/'gallery_3d.png', dpi=130, bbox_inches='tight'); plt.show()

## 6. Thesis summary CSV + zip

In [ ]:
all_runs = pd.concat([macro_df, br_df, size_df, container_df, edge_df], ignore_index=True)
summary = pd.DataFrame({
    'macro_util':   macro_df.groupby('algorithm')['util_pct'].mean(),
    'macro_std':    macro_df.groupby('algorithm')['util_pct'].std(),
    'br_util':      br_df.groupby('algorithm')['util_pct'].mean(),
    'edge_util':    edge_df.groupby('algorithm')['util_pct'].mean(),
    'cog_long_abs': macro_df.groupby('algorithm')['cog_long_dev'].apply(lambda s: s.abs().mean()),
    'imdg_total':   all_runs.groupby('algorithm')['imdg'].sum(),
    'unstable_tot': all_runs.groupby('algorithm')['unstable'].sum(),
    'mean_ms':      macro_df.groupby('algorithm')['elapsed_ms'].mean(),
}).reindex(ALL_ALGS).round(2)
summary.index = [ALG_LABELS[c] for c in summary.index]
summary.to_csv(OUT/'thesis_summary.csv')
print(summary.to_string())

import shutil
zip_path = '/kaggle/working/eval_results.zip' if os.path.isdir('/kaggle/working') else 'eval_results.zip'
shutil.make_archive(zip_path[:-4], 'zip', OUT)
print('zipped at', zip_path)